In [0]:
jdbc_hostname = "ep-little-pine-a9nunwps-pooler.gwc.azure.neon.tech"
jdbc_port = 5432
jdbc_database = "neondb"
jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}?sslmode=require&channelBinding=require"

connection_properties = {
    "user": "neondb_owner",
    "password": "npg_OpCX5QZW3UcD",
    "driver": "org.postgresql.Driver"
}


In [0]:
df = spark.read.jdbc(
    url=jdbc_url,
    table="public.customers",   
    properties=connection_properties
)
 
df.show()

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
databases = spark.catalog.listDatabases()

total_tables = 0
for db in databases:
    tables = spark.catalog.listTables(db.name)
    print(f"Database: {db.name}, Tables: {len(tables)}")
    total_tables += len(tables)

print(f"\nTotal number of tables in Databricks: {total_tables}")


In [0]:
customers_df = spark.table("customers")


In [0]:
from pyspark.sql.functions import col

filtered_df = df.filter(col("signup_date") > "2023-12-31")


In [0]:
filtered_df.write.mode("overwrite").saveAsTable("customers_after_2024")


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import lit
from datetime import datetime

# Create a new row manually (you can modify the values)
new_row = Row(
    customer_id=199,
    first_name="Aarya",
    last_name="Raman",
    email="aarya.raman@example.com",
    phone="+91-9876543210",
    city="Bengaluru",
    country="India",
    signup_date=datetime.strptime("2025-01-15", "%Y-%m-%d")
)

# Create a DataFrame with just that one row
new_row_df1 = spark.createDataFrame([new_row])


In [0]:
merged_df = filtered_df.unionByName(new_row_df1)
merged_df.write.mode("overwrite").saveAsTable("customers_after_2024")

In [0]:
combined_df = filtered_df.unionByName(new_row_df1)

# Overwrite the existing table with new row included



In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "customers_after_2024")

delta_table.alias("t").merge(
    new_row_df1.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

In [0]:
%%sql
CREATE OR REPLACE TABLE shallow_clone_customers
SHALLOW CLONE customers_after_2024


In [0]:
%%sql
CREATE OR REPLACE TABLE deep_clone_customers
DEEP CLONE customers_after_2024;


In [0]:
combined_df.write.mode("overwrite").saveAsTable("customers_after_2025")

In [0]:
%sql
DESCRIBE HISTORY customers_after_2024;


In [0]:
%sql
CREATE OR REPLACE TABLE customers_after_2024_v1
DEEP CLONE customers_after_2024 VERSION AS OF 0;


In [0]:
%sql
CREATE TABLE new_customer_table (
  customer_id STRING,
  name STRING,
  signup_date DATE
)
USING DELTA
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'id'
);


In [0]:
%sql
ALTER TABLE customers_after_2024 UPGRADE TO Delta 1;


In [0]:
pip install databricks-sdk
